In [ ]:
from src.server.models.dms_model import DmsModel
from src.db import DbConnection
from src.external_api.openai_api import chat_gpt4

prompts_dms_guid = '38506c50-b1e4-4017-84f0-a2a52fa1465b'
conn = DbConnection()
sheet_data = DmsModel.get_dms_data_to_json(conn, guid=prompts_dms_guid)
prompts = []
for it in sheet_data:
    if it['Agent'] == 'GPT4' and it['Type'] == 'Live':
        prompts.append({'area': it['Area'], 'prompt': it['Prompt']})

apps = [
    {'app_id': 'com.pixelfederation.ts2', 'title': 'Train Station 2'},
    {'app_id': 'tycoon.building.simulator.games.aldagames.com', 'title': 'Transport Tycoon Empire: City'}
]

for app in apps:
    for prompt in prompts:
        print(f"{app['title']}: {prompt['area']}")
        msg = prompt['prompt'].replace("[GAME TITLE]", app['title'])

        res = chat_gpt4(msg)

        conn.query("""
        INSERT INTO google_play_scraper.gpt_prompts_bckp (app_id, title, area, result)
        VALUES (%s, %s, %s, %s)
        """, [app['app_id'], app['title'], prompt['area'], res])

conn.commit()
conn.close()

In [ ]:
from src.external_api import write_sheet
from src.server.models.dms_model import DmsModel
from src.db import DbConnection

prompts_dms_guid = '38506c50-b1e4-4017-84f0-a2a52fa1465b'
conn = DbConnection()
sheet_data = DmsModel.get_dms_data_to_json(conn, guid=prompts_dms_guid)

rows = conn.select_all("""
SELECT * FROM google_play_scraper.gpt_prompts_bckp
""")


res = {
    'Title': ['Train Station 2', 'Transport Tycoon Empire: City']
}

for area in ['Genres', 'Mechanics', 'Content']:
    for it in sheet_data[area]:
        if f"{area}: {it['Category']}" not in res:
            res[f"{area}: {it['Category']}"] = [[], []]

#print(res)

for row in rows:
    r: str = row['result']
    area = row['area']
    title = row['title']
    r_a = r.split(",")
    rp = {}
    index = res['Title'].index(title)

    for i in range(len(r_a)):
        for k in res:
            if f"{area}: {r_a[i]}" == k:
                res[k][index].append(r_a[i + 1])

    #print(rp)
    #break


conn.close()
print(res)

values = [[], [], []]
for k in res:
    values[0].append(k)
    if not isinstance(res[k][0], str):
        if len(res[k][0]) == 0:
            values[1].append('-')
        else:
            values[1].append(','.join(res[k][0]))
    else:
        values[1].append(res[k][0])
    if not isinstance(res[k][1], str):
        if len(res[k][1]) == 0:
            values[2].append('-')
        else:
            values[2].append(','.join(res[k][1]))
    else:
        values[2].append(res[k][1])

#values

write_sheet("1TIu0NXPH8LFgVmlzA4-y2GdI80MbnQW_-FAvAxDCvys", "TS + Tycoon", values)


In [ ]:
from src.server.models.dms_constants import DmsConstants
from src.runtime_constants import rr
from src.server.models.dms_model import DmsModel
from src import external_api
from src.db import DbConnection

conn = DbConnection(rr.ENV_PROD)
sheet_data = {}
sheet_data['Prompts'] = external_api.read_sheet('1P__GHUVGUcnmRTUA1L7uF4CNkZD2VN7FLty9tS-4ltI', 'Prompts', True)
sheet_data['Genres'] = external_api.read_sheet('1P__GHUVGUcnmRTUA1L7uF4CNkZD2VN7FLty9tS-4ltI', 'Genres', True)
sheet_data['Mechanics'] = external_api.read_sheet('1P__GHUVGUcnmRTUA1L7uF4CNkZD2VN7FLty9tS-4ltI', 'Mechanics', True)
sheet_data['Content'] = external_api.read_sheet('1P__GHUVGUcnmRTUA1L7uF4CNkZD2VN7FLty9tS-4ltI', 'Content', True)
DmsModel.save_json_to_dms(conn, sheet_data, DmsConstants.prompts_guid, 'gpt prompts')
conn.commit()
conn.close()
sheet_data

In [ ]:
import openai
from src.project_paths import pp

model = "gpt-4"
messages = [
    {"role": "system", "content": """
You are a helpful market analyst who will tag the following mobile game: [GAME TITLE], based on the provided tagging key and store description. Here is the store description: '[STORE DESCRIPTION]'
"""},
    {"role": "user", "content": """
Please tag the qualities of the content of the following mobile game: [GAME TITLE], based on the provided store description and tagging key. Tagging key contains 'Category,Tag,(examples)'. In the output include all categories from the tagging key, but only applicable tags without any examples in brackets. Your output should look like this: '- Category: Tag1, Tag2, Tag3'. Here is the tagging key: 'Value,Contextual,(rewards, features),Value,Emotional,(nostalgic, cultural),Value,Expressive,(custom avatars, art),Value,Prestigious,(status, achievements),Value,Economic,(NFTs, tradeables),Value,Rarity,(rarity levels),Generation,Curated,(linear, single-player),Generation,Procedural,(programatic, AI),Generation,Customizable,(custom levels, skins),Generation,Player,(sandbox, modding),Variety,Low,(word, minimalist),Variety,Medium,(strategies, platformers),Variety,High,(sandbox, open-world RPGs),Progression,Low,(hypercasual, minimalist),Progression,Medium,(strategies, simulations),Progression,High,(open-world RPGs),Narratives,Adventure,(exploration, risk),Narratives,Mystery,(secrets, discoveries),Narratives,Fantasy,(made-up, magical),Narratives,Realistic,(true-to-life, historic),Narratives,Sci-Fi,(future, technology),Narratives,Action,(escapes, excitement),Narratives,Horror,(evil, fear),Narratives,Drama,(emotions, family),Narratives,Romance,(love, passion),Narratives,Comedy,(humor, satire),Narratives,None,(no narrative),Characters,Flat,(basic, static),Characters,Supporting,(intermediate, minor),Characters,Developed,(complex, dynamic),Characters,Few,(< 5 characters),Characters,Some,(5 - 15 characters),Characters,Many,(> 20 characters),Conflicts,Basic,(good/evil, disagreement),Conflicts,Intermediate,(dispute, moral dilemma),Conflicts,Complex,(geopolitical, mafia),Conflicts,Internal,(growth, fears),Conflicts,Interpersonal,(family, friends),Conflicts,Group,(competitor, villain),Conflicts,Societal,(issues, valuess),Conflicts,Global,(races, religions),Entities,Super-,(superpowered),Entities,Microbe,(cell, pathogen),Entities,Plant,(flower, tree),Entities,Animal,(pet, insect),Entities,Human,(regular),Entities,Celestial,(god, demigod),Entities,Monster,(zombie, daemon),Entities,Alien,(extraterrestrial),Entities,Robot,(machine, AI),Entities,Kid,(young, untrained),Entities,Adult,(mature, trained),Entities,Elder,(old, wise),Roles,Team,(multiple people),Roles,Authority,(leader, governor),Roles,Rebel,(outlaw, criminal),Roles,Creator,(builder, artist),Roles,Solver,(specialist, mentor),Roles,Mystic,(believer, mage),Roles,Warrior,(fighter, strong),Roles,Follower,(commoner, weak),Roles,Explorer,(adventurer, observer),Roles,Villain,(adversary, egoistic),Roles,Carer,(hero, altruistic),Eras,Prehistoric,(cavemen, dinosaurs),Eras,Ancient,(pyramids, Rome),Eras,Medieval,(knights, castles),Eras,Renaissance,(art, classics),Eras,Industrial,(engines, urbanization),Eras,Retro,(twenties to seventies),Eras,Modern,(internet, globalization),Eras,Near-future,(augmentation, technology),Eras,Far-future,(interstellar, aliens),Eras,Timeless,(any era, progress),Environment,Urban,(crowds, skyscrapers),Environment,Rural,(farms, fields),Environment,Suburban,(hoods, parks),Environment,Wilderness,(forests, deserts),Environment,Aquatic,(reefs, deep-sea),Environment,Space,(planets, nebulae),Environment,Alternality,(worlds, timelines),Environment,Virtuality,(digital, simulated),Domains,Politics,(power, conspiracy),Domains,Crime,(police, crime),Domains,Religion,(faith, spirituality),Domains,Mythology,(legends, traditions),Domains,Rights,(equality, minorities),Domains,Nature,(plants, animal),Domains,Family,(generations, friends),Domains,Sports,(team, atheltics),Domains,Military,(weapons, wars),Domains,Health,(mental, physical),Domains,Food,(cooking, recipes),Domains,Business,(company, wealth),Domains,Luxury,(cars, estates),Domains,Fashion,(beauty, design),Domains,Art,(works, music),Focus,Bases,(camps, headquarters),Focus,Cities,(buildings, quarters),Focus,Structures,(worlds, environments),Focus,Interiors,(interiors, rooms),Focus,Levels,(layouts, puzzles),Focus,Vehicles,(cars, trains),Focus,Weapons,(guns, swords),Focus,Objects,(other items),Focus,Solutions,(technology, science),Focus,Events,(story, history),Focus,Actors,(characters, animals),Focus,Self,(avatar, inventory),Issues,Disaster,(enviro, humanitarian),Issues,Violence,(harm, brutality),Issues,Exploitation,(material, sexual),Issues,Rejection,(discrimination, hate),Issues,Routine,(boredom, helplessness),Issues,Addiction,(substance abuse),Issues,Illness,(physical, mental),Issues,Inequality,(poverty, unemployment),Issues,Obsolence,(technical, personal),Educational,Low,(arcade, hypercasual),Educational,Medium,(puzzle, strategies),Educational,High,(edu games, documentaries),Visuals,High,(detailed, intricate),Visuals,Medium,(balanced art),Visuals,Low,(simple, minimal),Visuals,2D,(2D sprites, drawn),Visuals,3D,(3D models, rendered),Visuals,Top-down,(birds eye, parallax),Visuals,Isometric,(2.5D, strategy),Visuals,Side,(side-scrolling, platformer),Visuals,First-person,(shooter, racing),Visuals,Third-person,(RPG, action),Styles,Realistic,(photorealistic, true),Styles,Cartoon,(stylized, exaggerated),Styles,Anime,(Japanese, expressive),Styles,Pixel,(8 or 16-bit),Styles,Voxel,(3D, blocky),Consistency,Low,(hypercasual, arcades),Consistency,Medium,(puzzles, strategies),Consistency,High,(narrated, open-world RPGs),Language,Countries'
    """}
]

for i in range(len(messages)):
    messages[i]["content"] = messages[i]["content"].replace("[GAME TITLE]", "Train Station 2: Railroad Game").replace("[STORE DESCRIPTION]", """
All the railway tycoon lovers, train tycoon collectors and train strategy enthusiasts who love everything rail
transport related unite! It’s time to put your trains on rails and build a global railway empire. Become a railway
tycoon and enjoy a beautiful train strategy journey full of surprises, achievements and challenging contracts.
Discover and collect hundreds of famous real-life trains. It can become quite challenging sometimes, but as a railway
tycoon looking to build the biggest railway empire in the train world you’ll find a way. Develop your train station and
produce various products as some contractors may ask for something more than just raw materials. Upgrading trains
comes in handy because an upgraded train fleet is way more powerful when completing contracts and can save a lot of time!
Did you know that there are train rarity categories? The rarer the train, the higher the hauling capacity. Bending the
strategy tycoon rules has never been easier.
Train Station 2 game tycoon features:
- Own the most popular trains from railroad history of transport
- Collect famous express trains, upgrade them and reach their full transport potential
- Meet interesting simulator contractors and complete rail engine logistics jobs
- Coordinate and transport your trains according to your own simulator strategy
- Enhance your railway city and build bigger and better rail facilities to fit more trains
- Explore new global regions as your trains travel on railroad through city and land
- Play new events every month in the Train Station 2 strategy simulator
- Team up with fellow train tycoon enthusiasts during events & work together
- Compete in leaderboards to become the biggest railway tycoon
- Dispatch engines to collect resources and transport those to your contractors & city to complete train simulator jobs
Are you up for a challenge to collect the most trains, to build and manage a global trains empire and become the biggest
railway tycoon in the Train Station 2 world? Have you encountered a strategy railway contract which does not suit you at the moment?
Say no more! You can easily swap the contract requirements to get a better fit. PLEASE NOTE! Train Station 2 is an online free strategy
tycoon simulator game to download and play that requires a network connection to play. Some in-game items can also be purchased for real money.
if you don't want to use this feature please disable in-app purchases in your device's settings. Do you have any suggestions or problems
in your train station? Our caring community train managers would love to hear from you, visit https://care.pxfd.co/trainstation2!
Terms of Use: http://pxfd.co/eula
Privacy Policy: http://pxfd.co/privacy
Do you enjoy our 3D tycoon simulator game? Follow @TrainStation2 on social media to get the latest news and updates.
    """)

openai.api_key = pp.get_private_data()['open_ai']['chat_gpt4']['key']

completion = openai.ChatCompletion.create(
    model=model,
    messages=messages,
    temperature=0
)

print(completion.choices[0].message.content)

import pyperclip
pyperclip.copy(completion.choices[0].message.content)